# V12: Stacking Ensemble - Image2Biomass

This notebook combines predictions from V4 (EfficientNetV2), V7 (DINOv2), and V8 (DINOv2 + Depth) using a Ridge regression meta-learner.

**OOF R²: 0.90** (may overfit - V8 alone scored 0.62 LB)

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import pickle
import json

# Install dependencies if needed
!pip install -q timm albumentations transformers

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from transformers import AutoImageProcessor, AutoModelForDepthEstimation

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Paths
MODEL_DIR = Path('/kaggle/input/stacking-ensemble-v12/stacking_ensemble')
TEST_DIR = Path('/kaggle/input/csiro-biomass/test')
TEST_CSV = Path('/kaggle/input/csiro-biomass/test.csv')

TARGET_NAMES = ['Dry_Clover_g', 'Dry_Dead_g', 'Dry_Green_g', 'Dry_Total_g', 'GDM_g']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## Model Definitions

In [ ]:
class DepthEstimator(nn.Module):
    """Depth Anything v2 wrapper."""
    def __init__(self, model_path):
        super().__init__()
        self.processor = AutoImageProcessor.from_pretrained(model_path)
        self.model = AutoModelForDepthEstimation.from_pretrained(model_path)
        self.model.eval()
        for param in self.model.parameters():
            param.requires_grad = False
    
    def forward(self, x):
        # x: [B, 3, H, W] normalized tensor
        # Denormalize
        mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(x.device)
        std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(x.device)
        x_denorm = x * std + mean
        x_denorm = torch.clamp(x_denorm, 0, 1)
        
        with torch.no_grad():
            outputs = self.model(pixel_values=x_denorm)
            depth = outputs.predicted_depth
        
        # Normalize depth
        depth = depth.unsqueeze(1)
        depth_min = depth.amin(dim=(2, 3), keepdim=True)
        depth_max = depth.amax(dim=(2, 3), keepdim=True)
        depth = (depth - depth_min) / (depth_max - depth_min + 1e-8)
        
        # Resize to match input
        depth = nn.functional.interpolate(depth, size=x.shape[2:], mode='bilinear', align_corners=False)
        return depth

In [ ]:
class MultiTaskEfficientNet(nn.Module):
    """V4 architecture - EfficientNetV2."""
    def __init__(self, dropout=0.5, head_hidden_dim=512):
        super().__init__()
        self.target_names = TARGET_NAMES
        self.backbone = timm.create_model('tf_efficientnetv2_m', pretrained=False, num_classes=0, global_pool='avg')
        num_features = self.backbone.num_features
        
        self.heads = nn.ModuleDict()
        for name in self.target_names:
            self.heads[name] = nn.Sequential(
                nn.Linear(num_features, head_hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(head_hidden_dim, 1)
            )
    
    def forward(self, x):
        features = self.backbone(x)
        return {name: self.heads[name](features).squeeze(-1) for name in self.target_names}


class FoundationModel(nn.Module):
    """V7 architecture - DINOv2."""
    def __init__(self, num_features=768, dropout=0.3):
        super().__init__()
        self.target_names = TARGET_NAMES
        self.backbone = timm.create_model('vit_base_patch14_dinov2', pretrained=False, num_classes=0)
        
        self.heads = nn.ModuleDict()
        for name in self.target_names:
            self.heads[name] = nn.Sequential(
                nn.Linear(num_features, 256),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(256, 64),
                nn.GELU(),
                nn.Linear(64, 1)
            )
    
    def forward(self, x):
        features = self.backbone(x)
        return {name: self.heads[name](features).squeeze(-1) for name in self.target_names}


class FoundationModelWithDepth(nn.Module):
    """V8 architecture - DINOv2 + Depth."""
    def __init__(self, depth_model_path, num_features=768, dropout=0.3):
        super().__init__()
        self.target_names = TARGET_NAMES
        self.backbone = timm.create_model('vit_base_patch14_dinov2', pretrained=False, num_classes=0)
        self.depth_estimator = DepthEstimator(depth_model_path)
        
        self.depth_encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm2d(32),
            nn.GELU(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.GELU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.GELU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        fused_features = num_features + 256
        self.heads = nn.ModuleDict()
        for name in self.target_names:
            self.heads[name] = nn.Sequential(
                nn.Linear(fused_features, 256),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(256, 64),
                nn.GELU(),
                nn.Linear(64, 1)
            )
    
    def forward(self, x):
        rgb_features = self.backbone(x)
        with torch.no_grad():
            depth_maps = self.depth_estimator(x)
        depth_features = self.depth_encoder(depth_maps)
        fused = torch.cat([rgb_features, depth_features], dim=1)
        return {name: self.heads[name](fused).squeeze(-1) for name in self.target_names}

## Load Models

In [ ]:
def get_transform(image_size):
    return A.Compose([
        A.Resize(image_size, image_size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ])

def load_image(image_path, transform, device):
    image = Image.open(image_path).convert('RGB')
    image_np = np.array(image)
    transformed = transform(image=image_np)
    return transformed['image'].unsqueeze(0).to(device)

def predict_with_model(model, image_tensor):
    with torch.no_grad():
        outputs = model(image_tensor)
        return {name: outputs[name].cpu().numpy()[0] for name in TARGET_NAMES}

In [ ]:
# Load meta-learners
print("Loading meta-learners...")
with open(MODEL_DIR / 'meta_learners.pkl', 'rb') as f:
    meta_learners = pickle.load(f)

with open(MODEL_DIR / 'config.json', 'r') as f:
    config = json.load(f)

model_names = config['model_names']
print(f"Models: {model_names}")

In [ ]:
# Load test data
print("\nLoading test data...")
test_df = pd.read_csv(TEST_CSV)
test_df['image_id'] = test_df['sample_id'].str.split('__').str[0]
test_image_ids = test_df['image_id'].unique()
print(f"Test images: {len(test_image_ids)}")

## Generate Predictions

In [ ]:
# V4 predictions
print("\n" + "="*60)
print("V4 (EfficientNetV2) predictions")
print("="*60)

v4_transform = get_transform(512)
v4_preds = {target: np.zeros(len(test_image_ids)) for target in TARGET_NAMES}
n_v4_models = 0

for fold_idx in range(5):
    checkpoint_path = MODEL_DIR / 'v4_checkpoints' / f'fold_{fold_idx}_best_model.pth'
    if not checkpoint_path.exists():
        print(f"Fold {fold_idx}: Not found")
        continue
    
    model = MultiTaskEfficientNet(dropout=0.5, head_hidden_dim=512)
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval()
    n_v4_models += 1
    print(f"Fold {fold_idx}: Loaded")
    
    for idx, image_id in enumerate(tqdm(test_image_ids, desc=f"Fold {fold_idx}")):
        image_path = TEST_DIR / f"{image_id}.jpg"
        image_tensor = load_image(image_path, v4_transform, device)
        preds = predict_with_model(model, image_tensor)
        for target in TARGET_NAMES:
            v4_preds[target][idx] += preds[target]
    
    del model
    torch.cuda.empty_cache()

for target in TARGET_NAMES:
    v4_preds[target] /= n_v4_models
print(f"Averaged {n_v4_models} folds")

In [ ]:
# V7 predictions
print("\n" + "="*60)
print("V7 (DINOv2) predictions")
print("="*60)

v7_transform = get_transform(518)
v7_preds = {target: np.zeros(len(test_image_ids)) for target in TARGET_NAMES}
n_v7_models = 0

for fold_idx in range(5):
    checkpoint_path = MODEL_DIR / 'v7_checkpoints' / f'fold_{fold_idx}_best_model.pth'
    if not checkpoint_path.exists():
        print(f"Fold {fold_idx}: Not found")
        continue
    
    model = FoundationModel(num_features=768, dropout=0.3)
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval()
    n_v7_models += 1
    print(f"Fold {fold_idx}: Loaded")
    
    for idx, image_id in enumerate(tqdm(test_image_ids, desc=f"Fold {fold_idx}")):
        image_path = TEST_DIR / f"{image_id}.jpg"
        image_tensor = load_image(image_path, v7_transform, device)
        preds = predict_with_model(model, image_tensor)
        for target in TARGET_NAMES:
            v7_preds[target][idx] += preds[target]
    
    del model
    torch.cuda.empty_cache()

for target in TARGET_NAMES:
    v7_preds[target] /= n_v7_models
print(f"Averaged {n_v7_models} folds")

In [ ]:
# V8 predictions
print("\n" + "="*60)
print("V8 (DINOv2 + Depth) predictions")
print("="*60)

depth_model_path = MODEL_DIR / 'depth_anything_v2'
v8_transform = get_transform(518)
v8_preds = {target: np.zeros(len(test_image_ids)) for target in TARGET_NAMES}
n_v8_models = 0

for fold_idx in range(5):
    checkpoint_path = MODEL_DIR / 'v8_checkpoints' / f'fold_{fold_idx}_best_model.pth'
    if not checkpoint_path.exists():
        print(f"Fold {fold_idx}: Not found")
        continue
    
    model = FoundationModelWithDepth(depth_model_path, num_features=768, dropout=0.3)
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval()
    n_v8_models += 1
    print(f"Fold {fold_idx}: Loaded")
    
    for idx, image_id in enumerate(tqdm(test_image_ids, desc=f"Fold {fold_idx}")):
        image_path = TEST_DIR / f"{image_id}.jpg"
        image_tensor = load_image(image_path, v8_transform, device)
        preds = predict_with_model(model, image_tensor)
        for target in TARGET_NAMES:
            v8_preds[target][idx] += preds[target]
    
    del model
    torch.cuda.empty_cache()

for target in TARGET_NAMES:
    v8_preds[target] /= n_v8_models
print(f"Averaged {n_v8_models} folds")

## Apply Meta-Learner

In [ ]:
print("\n" + "="*60)
print("Applying Meta-Learner")
print("="*60)

# Combine predictions
test_predictions = {
    'V4_EfficientNet': v4_preds,
    'V7_DINOv2': v7_preds,
    'V8_DINOv2_Depth': v8_preds
}

# Apply meta-learner
final_predictions = {}
for target in TARGET_NAMES:
    X = np.column_stack([
        test_predictions[model_name][target]
        for model_name in model_names
    ])
    final_predictions[target] = meta_learners[target].predict(X)
    print(f"{target}: mean={final_predictions[target].mean():.2f}, std={final_predictions[target].std():.2f}")

In [ ]:
# Enforce biological constraints
print("\nEnforcing biological constraints...")

for idx in range(len(test_image_ids)):
    green = final_predictions['Dry_Green_g'][idx]
    clover = final_predictions['Dry_Clover_g'][idx]
    dead = final_predictions['Dry_Dead_g'][idx]
    gdm = final_predictions['GDM_g'][idx]
    total = final_predictions['Dry_Total_g'][idx]
    
    # Ensure non-negative
    green = max(0, green)
    clover = max(0, clover)
    dead = max(0, dead)
    
    # Soft constraint: GDM = Green + Clover
    expected_gdm = green + clover
    adjusted_gdm = (gdm + expected_gdm) / 2
    
    # Soft constraint: Total = GDM + Dead
    expected_total = adjusted_gdm + dead
    adjusted_total = (total + expected_total) / 2
    
    final_predictions['Dry_Green_g'][idx] = green
    final_predictions['Dry_Clover_g'][idx] = clover
    final_predictions['Dry_Dead_g'][idx] = dead
    final_predictions['GDM_g'][idx] = max(0, adjusted_gdm)
    final_predictions['Dry_Total_g'][idx] = max(0, adjusted_total)

print("Constraints applied.")

## Create Submission

In [ ]:
# Create submission DataFrame
submissions = []
for idx, image_id in enumerate(test_image_ids):
    for target in TARGET_NAMES:
        sample_id = f"{image_id}__{target}"
        value = final_predictions[target][idx]
        submissions.append({
            'sample_id': sample_id,
            'target': value
        })

submission_df = pd.DataFrame(submissions)
submission_df.to_csv('submission.csv', index=False)

print(f"\nSubmission saved!")
print(f"Total rows: {len(submission_df)}")
print(f"\nPrediction statistics:")
for target in TARGET_NAMES:
    values = final_predictions[target]
    print(f"  {target}: mean={values.mean():.2f}, std={values.std():.2f}, min={values.min():.2f}, max={values.max():.2f}")

In [ ]:
# Preview
submission_df.head(10)